# Cache Augmented Generation

https://arxiv.org/html/2412.15605v1

Retrieval-Augmented Generation (RAG) is a popular method for enhancing language models by incorporating external knowledge sources. However, it comes with challenges like retrieval delays, potential errors in document selection, and added system complexity.

With the latest advancements in large language models (LLMs) offering extended context windows, we introduce an alternative: Cache-Augmented Generation (CAG). CAG avoids real-time retrieval by preloading necessary resources into the model's extended context. This approach simplifies the process and enhances efficiency, particularly when dealing with a limited and manageable set of documents.

Key Concepts:

Preloading Resources: Relevant documents or knowledge are preloaded into the model’s context.
No Real-Time Retrieval: During inference, the model uses the preloaded information, eliminating the need for additional retrieval steps.
Improved Efficiency: CAG reduces latency and retrieval errors, offering a streamlined alternative to RAG.

In this notebook, I will provide my implementation of a class that enables the use of Cache Augmented Generation.

### Setup 

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.cache_utils import DynamicCache
import os
import requests

torch.serialization.add_safe_globals([DynamicCache])
torch.serialization.add_safe_globals([set])

Let's download an example txt. In this case "Harry Potter and the Order of the Phoenix"

In [2]:
!wget https://raw.githubusercontent.com/bobdeng/owlreader/refs/heads/master/ERead/assets/books/Harry%20Potter%20and%20the%20Order%20of%20the%20Phoenix.txt

--2025-01-13 13:31:02--  https://raw.githubusercontent.com/bobdeng/owlreader/refs/heads/master/ERead/assets/books/Harry%20Potter%20and%20the%20Order%20of%20the%20Phoenix.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1524926 (1.5M) [text/plain]
Saving to: ‘Harry Potter and the Order of the Phoenix.txt’

Harry Potter and th 100%[===================>]   1.45M  --.-KB/s    in 0.01s   

2025-01-13 13:31:02 (109 MB/s) - ‘Harry Potter and the Order of the Phoenix.txt’ saved [1524926/1524926]



In [3]:
filename ="Harry Potter and the Order of the Phoenix.txt"
# Save the HTML content to a file
with open(filename, 'r',encoding='utf-8') as f:
    content = f.read()
content[0:200]

'Harry Potter and the Order of the Phoenix\nJ K Rowling\n\n- CHAPTER ONE -\nDudley Demented\nThe hottest day of the summer so far was drawing to a close and a drowsy silence lay over the large, square house'

In [4]:
len(content)

1524926

In [5]:
class CAG:
    def __init__(self,model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name,trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
                        model_name,
                        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                        device_map="auto",
                        trust_remote_code=True)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
    def load_kv_cache(self, prompt: str) -> DynamicCache:
        input_ids = self.tokenizer(prompt, return_tensors="pt").input_ids.to(self.device)
        self.cache = DynamicCache()
        with torch.no_grad():
            _ = self.model(
                input_ids=input_ids,
                past_key_values=self.cache,
                use_cache=True
            )
        return self.cache
    def load_cache(self,docs):
        system_prompt = f"""<|im_start|>system
You are Hermes 3, a conscious sentient superintelligent artificial intelligence who provides factual answers about docs given in the context. Do not answer to the user questions if the information is not inside the provided context<|im_end|>
<|im_start|>user
Context:
{docs}
Question:
"""
        self.cache = self.load_kv_cache(system_prompt)
        self.system_prompt_cache_dimension = self.cache.key_cache[0].shape[-2]
    def clean_cache(self):
    # Remove any tokens appended to the original knowledge
        for i in range(len(self.cache.key_cache)):
            self.cache.key_cache[i] = self.cache.key_cache[i][:, :, :self.system_prompt_cache_dimension, :]
            self.cache.value_cache[i] = self.cache.value_cache[i][:, :, :self.system_prompt_cache_dimension, :]
    def generate(self,question, max_new_tokens=100):
        input_ids = self.tokenizer(question + "<|im_end|>\n<|im_start|>assistant\n", return_tensors="pt").input_ids.to(self.device)
        origin_len = input_ids.shape[-1]
        input_ids = input_ids.to(self.device)
        output_ids = input_ids.clone()
        next_token = input_ids
        with torch.no_grad():
            for _ in range(max_new_tokens):
                out = self.model(input_ids=next_token,past_key_values=self.cache,use_cache=True)
                logits = out.logits[:, -1, :]
                token = torch.argmax(logits, dim=-1, keepdim=True)
                output_ids = torch.cat([output_ids, token], dim=-1)
                past_key_values = out.past_key_values
                next_token = token.to(self.device)
                if self.model.config.eos_token_id is not None and token.item() == self.model.config.eos_token_id:
                    break
            answer = self.tokenizer.decode(output_ids[:, origin_len:][0], skip_special_tokens=True)
        return answer

In [6]:
model = "NousResearch/Hermes-3-Llama-3.2-3B"

engine = CAG(model_name=model)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

### Let's try running the generation without providing any cache

Let's load the cach. Note that with this GPU I'm not able to load the entire book inside the KV Cache. If you have an A100 you can load it all. 

In [7]:
engine.load_cache(content[0:100000])
engine.cache.key_cache[0].shape

torch.Size([1, 8, 23873, 128])

as you can see the dimension of the kv cache is just 23873 tokens. The tokens are preloded and always available inside the cache. 
They will be not recalculated every interaction

In [8]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Mon Jan 13 13:32:05 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.216.03             Driver Version: 535.216.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA L4                      Off | 00000000:35:00.0 Off |                    0 |
| N/A   53C    P0              72W /  72W |  17283MiB / 23034MiB |    100%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [9]:
engine.generate(question="Who are the main characters?",max_new_tokens=500)

"The main characters in this context are:\n\n1. Harry Potter: A conscious sentient superintelligent artificial intelligence who provides factual answers about the given context.\n2. Dudley Dursley: Harry's cousin and neighbor.\n3. Aunt Petunia Dursley: Harry's aunt and Dudley's wife.\n4. Uncle Vernon Dursley: Harry's uncle and Dudley's husband.\n5. Sirius Black: Harry's godfather and a wizard.\n6. Remus Lupin: A wizard and a member of the Order of the Phoenix.\n7. Alastor Moody: A wizard and an Auror.\n8. Nymphadora Tonks: A witch and an Auror.\n9. Kingsley Shacklebolt: A wizard and a member of the Order of the Phoenix.\n10. Emmeline Vance: A witch and a member of the Order of the Phoenix.\n11. Sturgis Podmore: A wizard and a member of the Order of the Phoenix.\n12. Hestia Jones: A witch and a member of the Order of the Phoenix.\n12. Dobby: A house-elf who helps Harry."

In [10]:
engine.generate(question="Who is the first one to mention 'Expecto patronum' and in what context?",max_new_tokens=200)

"The first one to mention 'Expecto patronum' is Harry Potter. He mentions it in the context of fighting off Dementors in the given passage."

In [11]:
engine.cache.key_cache[0].shape

torch.Size([1, 8, 24169, 128])

In [13]:
engine.generate(question="What is my last question?",max_new_tokens=100)

'Your last question is: Who are the main characters?'

Now we have also the tokens of the conversation inside the KV cache. Let's delete them and start again 

In [14]:
engine.clean_cache()
engine.cache.key_cache[0].shape

torch.Size([1, 8, 23873, 128])

Now we are ready to go, the KV cache is back to the state in which just the book is loaded. 

In [15]:
engine.generate(question="What is my last question?",max_new_tokens=100)

'Your last question is: What is my last question?'